# 06 - Despliegue y Consumo de la API

## Que hace este notebook?
Demuestra cómo consumir el modelo que acabamos de entrenar y evaluar
a través de la API REST que está corriendo en Docker.

## Prerequisitos
- Haber ejecutado los notebooks 01 a 05
- Docker corriendo con: `docker compose up -d`
- API disponible en: http://localhost:8000

## Que veremos?
1. Verificar que la API está activa
2. Hacer predicciones individuales
3. Comparar predicciones entre zonas distintas
4. Ver todos los endpoints disponibles
5. Cerrar el ciclo MLOps

## Prerequisitos
- Haber ejecutado: `05_evaluacion_final.ipynb`
- Docker corriendo: `docker compose up -d` (desde la raíz del proyecto)
- Requiere: API en ejecución en `http://localhost:8000`

## Arquitectura del despliegue

El servicio se despliega dentro de un **contenedor Docker**, lo que garantiza que funciona igual
en el equipo del desarrollador, en el servidor de staging y en producción.

```
Usuario de negocio  →  Web UI (localhost:8000/ui)
                              ↓
Aplicación externa  →  API REST (localhost:8000/v1/predecir)
                              ↓
                       FastAPI (src/serving/api.py)
                              ↓
                    modelo_produccion.pkl + scaler.pkl
                              ↓
                    Contenedor Docker (mlops-api-viviendas)
```

**Componentes clave:**
| Componente | Rol |
|---|---|
| `FastAPI` | Framework web de alto rendimiento para la API REST |
| `modelo_produccion.pkl` | Modelo entrenado y aprobado en etapas anteriores |
| `scaler.pkl` | Transformador de features para normalizar las entradas |
| `Docker` | Contenerización del servicio completo |
| `docker-compose.yml` | Orquestación y configuración del contenedor |

## 1. Levantar el servicio con Docker

El proyecto incluye `Dockerfile` y `docker-compose.yml` listos para usar.

Para iniciar el servicio, ejecuta desde la raíz del proyecto:

```bash
# Construir la imagen y levantar el contenedor
docker-compose up --build -d

# Ver logs del contenedor
docker-compose logs -f

# Detener el servicio
docker-compose down
```

In [1]:
# Verificar que Docker está corriendo
import subprocess
result = subprocess.run(
    ['docker', 'ps', '--filter', 'name=mlops-api-viviendas',
     '--format', 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'],
    capture_output=True, text=True
)
print("Contenedores activos:")
print(result.stdout if result.stdout else "No hay contenedores corriendo")

Contenedores activos:
NAMES                 STATUS                 PORTS
mlops-api-viviendas   Up 6 hours (healthy)   0.0.0.0:8000->8000/tcp



In [2]:
import requests

try:
    resp = requests.get("http://localhost:8000/health", timeout=5)
    if resp.status_code == 200:
        print("API en linea y modelo cargado")
        print(f"   Respuesta: {resp.json()}")
    else:
        print(f"API respondio con status {resp.status_code}")
except requests.exceptions.ConnectionError:
    print("La API no esta disponible.")
    print("   Ejecuta desde la raiz del proyecto:")
    print("   > docker compose up -d")
    print("   Espera unos segundos y vuelve a ejecutar esta celda.")

API en linea y modelo cargado
   Respuesta: {'status': 'healthy', 'modelo_cargado': True}


## 2. Verificar el estado del servicio (Health Check)

Antes de realizar predicciones, comprobamos que la API está en línea y el modelo está cargado.

**Endpoints disponibles:**

| Endpoint | Método | Descripción |
|---|---|---|
| `/` | GET | Confirmación básica de que el servicio responde |
| `/health` | GET | Estado detallado: modelo cargado, versión, métricas |
| `/v1/predecir` | POST | Recibe datos de una vivienda y devuelve el precio estimado |
| `/v1/info-modelo` | GET | Metadatos del modelo: algoritmo, features, fecha de entrenamiento |
| `/ui` | GET | Interfaz web visual para usuarios sin conocimientos técnicos |
| `/docs` | GET | Documentación interactiva Swagger UI |

> ⚠️ Asegúrate de que el contenedor Docker esté corriendo antes de ejecutar las siguientes celdas.

In [3]:
import requests
import json

BASE_URL = "http://localhost:8000"

# Health check
resp = requests.get(f"{BASE_URL}/health")
print("Estado de la API:")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

Estado de la API:
{
  "status": "healthy",
  "modelo_cargado": true
}


## Endpoint de Predicción: `POST /v1/predecir`

Este es el endpoint principal del servicio. Recibe los datos de una vivienda en formato JSON
y devuelve el precio estimado en dólares.

**Campos de entrada** (las mismas features del dataset California Housing):

| Campo | Tipo | Descripción |
|---|---|---|
| `MedInc` | float | Ingreso mediano del bloque (en decenas de miles de USD) |
| `HouseAge` | float | Años de antigüedad de la vivienda |
| `AveRooms` | float | Promedio de habitaciones por vivienda |
| `AveBedrms` | float | Promedio de dormitorios por vivienda |
| `Population` | float | Población del bloque |
| `AveOccup` | float | Promedio de ocupantes por vivienda |
| `Latitude` | float | Latitud geográfica |
| `Longitude` | float | Longitud geográfica |

**Respuesta:**
```json
{
  "precio_estimado_usd": 425000.0,
  "modelo_version": "1.0.0",
  "timestamp": "2024-01-15T10:30:00"
}
```

> Este es **el mismo modelo** entrenado con GradientBoosting en el notebook 03, seleccionado
> en el notebook 04 y aprobado en el gate de calidad del notebook 05.

In [4]:
# Ejemplo 1: Zona de alto valor (San Francisco)
vivienda_cara = {
    "MedInc": 8.5,
    "HouseAge": 15,
    "AveRooms": 7.0,
    "AveBedrms": 1.2,
    "Population": 800,
    "AveOccup": 2.5,
    "Latitude": 37.77,
    "Longitude": -122.42
}

resp = requests.post(f"{BASE_URL}/v1/predecir", json=vivienda_cara)
resultado = resp.json()
print("🏙️  Vivienda en zona de alto valor (San Francisco):")
print(f"   Precio estimado: ${resultado['precio_estimado_usd']:,.0f} USD")

🏙️  Vivienda en zona de alto valor (San Francisco):
   Precio estimado: $432,559 USD


In [5]:
# Ejemplo 2: Zona de menor valor (interior de California)
vivienda_economica = {
    "MedInc": 2.5,
    "HouseAge": 35,
    "AveRooms": 4.5,
    "AveBedrms": 1.1,
    "Population": 2000,
    "AveOccup": 3.5,
    "Latitude": 36.5,
    "Longitude": -119.5
}

resp = requests.post(f"{BASE_URL}/v1/predecir", json=vivienda_economica)
resultado = resp.json()
print("🏘️  Vivienda en zona económica (Valle Central):")
print(f"   Precio estimado: ${resultado['precio_estimado_usd']:,.0f} USD")

🏘️  Vivienda en zona económica (Valle Central):
   Precio estimado: $74,194 USD


In [6]:
import pandas as pd

# Comparativa de 5 viviendas con diferentes características
viviendas = [
    {"zona": "San Francisco (lujo)",    "MedInc": 8.5, "HouseAge": 15, "AveRooms": 7.0, "AveBedrms": 1.2, "Population": 800,  "AveOccup": 2.5, "Latitude": 37.77, "Longitude": -122.42},
    {"zona": "Los Angeles (premium)",    "MedInc": 6.5, "HouseAge": 20, "AveRooms": 6.0, "AveBedrms": 1.3, "Population": 1200, "AveOccup": 2.8, "Latitude": 34.05, "Longitude": -118.25},
    {"zona": "San Diego (medio)",         "MedInc": 4.5, "HouseAge": 25, "AveRooms": 5.0, "AveBedrms": 1.1, "Population": 1500, "AveOccup": 3.0, "Latitude": 32.72, "Longitude": -117.15},
    {"zona": "Sacramento (asequible)",    "MedInc": 3.2, "HouseAge": 30, "AveRooms": 4.5, "AveBedrms": 1.1, "Population": 1800, "AveOccup": 3.2, "Latitude": 38.58, "Longitude": -121.49},
    {"zona": "Valle Central (económico)", "MedInc": 2.5, "HouseAge": 35, "AveRooms": 4.5, "AveBedrms": 1.1, "Population": 2000, "AveOccup": 3.5, "Latitude": 36.5,  "Longitude": -119.5},
]

resultados = []
for v in viviendas:
    payload = {k: val for k, val in v.items() if k != 'zona'}
    resp = requests.post(f"{BASE_URL}/v1/predecir", json=payload)
    precio = resp.json()['precio_estimado_usd']
    resultados.append({
        "Zona": v["zona"],
        "Ingreso (MedInc)": v["MedInc"],
        "Edad (años)": v["HouseAge"],
        "Precio Estimado (USD)": f"${precio:,.0f}"
    })

df_comparativa = pd.DataFrame(resultados)
df_comparativa = df_comparativa.sort_values("Precio Estimado (USD)", ascending=False).reset_index(drop=True)
print("Comparativa de precios estimados por zona:")
df_comparativa

Comparativa de precios estimados por zona:


,Zona,Ingreso (MedInc),Edad (años),Precio Estimado (USD)
0,Valle Central (económico),2.5,35,"$74,194"
1,San Francisco (lujo),8.5,15,"$432,559"
2,Los Angeles (premium),6.5,20,"$325,317"
3,San Diego (medio),4.5,25,"$167,442"
4,Sacramento (asequible),3.2,30,"$125,108"


## 3. Interfaz Web — Para usuarios sin conocimientos técnicos

Además de la API REST (para integraciones técnicas), el servicio incluye una **interfaz web visual** 
diseñada para usuarios de negocio que no saben programar.

**Acceso:** [http://localhost:8000/ui](http://localhost:8000/ui)

La interfaz permite:
- Ingresar los datos de una vivienda mediante formulario
- Obtener el precio estimado con un clic
- Sin necesidad de conocer JSON ni programación

**Documentación interactiva (Swagger):** [http://localhost:8000/docs](http://localhost:8000/docs)

In [7]:
# Resumen de todos los endpoints disponibles
endpoints = [
    {"Endpoint": "GET /",             "Tipo": "Health check",      "Audiencia": "Sistemas/DevOps"},
    {"Endpoint": "GET /health",       "Tipo": "Estado del modelo",  "Audiencia": "Kubernetes/Load Balancer"},
    {"Endpoint": "POST /v1/predecir", "Tipo": "Predicción",          "Audiencia": "Desarrolladores/Apps"},
    {"Endpoint": "GET /v1/info-modelo","Tipo": "Info del modelo",   "Audiencia": "Data Scientists"},
    {"Endpoint": "GET /ui",           "Tipo": "Interfaz web",       "Audiencia": "Usuarios de negocio"},
    {"Endpoint": "GET /docs",         "Tipo": "Swagger UI",         "Audiencia": "Desarrolladores"},
]
import pandas as pd
df_endpoints = pd.DataFrame(endpoints)
df_endpoints

,Endpoint,Tipo,Audiencia
0,GET /,Health check,Sistemas/DevOps
1,GET /health,Estado del modelo,Kubernetes/Load Balancer
2,POST /v1/predecir,Predicción,Desarrolladores/Apps
3,GET /v1/info-modelo,Info del modelo,Data Scientists
4,GET /ui,Interfaz web,Usuarios de negocio
5,GET /docs,Swagger UI,Desarrolladores


## Ciclo Completo MLOps — Resumen de los 6 notebooks

Hemos recorrido todo el ciclo de vida de un modelo de Machine Learning en producción:

```
┌─────────────────────────────────────────────────────────────────┐
│                  CICLO COMPLETO RECORRIDO                        │
├────┬────────────────────────────┬───────────────────────────────┤
│ 01 │ Análisis Exploratorio       │ Entender los datos            │
│ 02 │ Ingeniería de Features      │ Preparar variables            │
│ 03 │ Entrenamiento + MLflow      │ Experimentar y registrar      │
│ 04 │ Selección de Modelo         │ Elegir el mejor               │
│ 05 │ Evaluación Final            │ Gate de calidad               │
│ 06 │ Despliegue API + Web UI     │ ← Estamos aquí                │
└────┴────────────────────────────┴───────────────────────────────┘
         ↑ Si el modelo degrada en producción → volver al inicio
```

### El ciclo no termina aquí: Monitoreo y Re-entrenamiento

El despliegue no es el fin del proceso MLOps. Una vez en producción, el modelo debe ser **monitoreado continuamente**:

- **Data Drift:** ¿cambió la distribución de los datos de entrada?
- **Concept Drift:** ¿la relación entre features y target ya no es válida?
- **Performance Degradation:** ¿las métricas en producción bajan del umbral aceptable?

Cuando se detecta degradación, el sistema **dispara automáticamente** un nuevo ciclo de entrenamiento,
pasando nuevamente por los notebooks 03 → 04 → 05 → 06. Así se cierra el ciclo MLOps completo.

```
Producción → Monitoreo → Drift detectado → Re-entrenamiento → Validación → Despliegue
    └────────────────────────────────────────────────────┘
```

In [8]:
print("=" * 60)
print("  CICLO MLOps COMPLETADO")
print("=" * 60)
print()
print("  01 EDA              -> Entendimos los datos")
print("  02 Features         -> Preparamos variables")
print("  03 MLflow           -> Experimentamos con 3 algoritmos")
print("  04 Seleccion        -> Elegimos el mejor modelo")
print("  05 Evaluacion       -> Validamos con datos nunca vistos")
print("  06 Despliegue       -> Consumimos el modelo en produccion")
print()
print("  El modelo esta disponible en: http://localhost:8000/ui")
print("  Documentacion API:            http://localhost:8000/docs")
print("=" * 60)

  CICLO MLOps COMPLETADO

  01 EDA              -> Entendimos los datos
  02 Features         -> Preparamos variables
  03 MLflow           -> Experimentamos con 3 algoritmos
  04 Seleccion        -> Elegimos el mejor modelo
  05 Evaluacion       -> Validamos con datos nunca vistos
  06 Despliegue       -> Consumimos el modelo en produccion

  El modelo esta disponible en: http://localhost:8000/ui
  Documentacion API:            http://localhost:8000/docs
